Extraction with GPT 4.0 ONLY (without BAML)

In [1]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

print("API key loaded:", os.getenv("OPENAI_API_KEY") is not None)

repo_root = Path.cwd()

if repo_root.name == "knowledge_extraction":
    repo_root = repo_root.parent.parent

input_file = repo_root / "data" / "processed" / "20_cases_for_baml_fewshot.jsonl"

print(input_file)
print("File exists:", input_file.exists())

cases = []

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            cases.append(json.loads(line))

print("Number of cases:", len(cases))
print(cases[0])

API key loaded: True
c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\processed\20_cases_for_baml_fewshot.jsonl
File exists: True
Number of cases: 20
Case-ID: IBM3_C15_23-Feb-15_23-Feb-15
Sources: storing
Issues:
- He-verlies/tekort in cryocompressor |  | Uit: 10,5bar  /  aan: 3,0 - 3,4bar  (40-50 psi) | Displacer maakt een wat hoog piepend/schrapend geluid | Temp. Is 11K |  | (Absorber vervangen: 6-10-'06) | CTI-Cryogenics | Helix technology Corp | 9600 compressor | P/N = 8135901G001 | SN = 94L0013912
Resolution Status: Unknown


In [ ]:
def build_prompt(log_text):
    return f"""
You are an expert in analyzing technical maintenance logs.

Extract the information from this case text and return only valid JSON.
Do not add any explanation, markdown, code fences, or extra fields.

Return a single JSON object with exactly these fields:
- "fault_location": {{"name": "...", "machine": "IBM3" or "IBM4" or null}}
- "fault_symptoms": array of strings
- "fault_reason": array of {{"name": string}} objects, or []
- "fault_measures": array of {{"description": string}} objects, or []
- "resolution_status": "Resolved", "Unknown", or "N/A"

If a field is not present, return an empty array or "Unknown" rather than omitting the field.

Example output format:
{{
  "fault_location": {{
    "name": "",
    "machine": null
  }},
  "fault_symptoms": [],
  "fault_reason": [],
  "fault_measures": [],
  "resolution_status": "Unknown"
}}

Maintenance log:
---
{log_text}
---
"""

In [21]:
output_results = []

for idx, case in enumerate(cases, start=1):
    print(f"Processing case {idx}/{len(cases)}")

    # Your jsonl may be either raw string or dict depending on preprocessing.
    if isinstance(case, str):
        log_text = case
        first_line = log_text.splitlines()[0].strip() if log_text else ""
        if first_line.startswith("Case-ID:"):
            case_id = first_line.replace("Case-ID:", "").strip()
        else:
            case_id = f"case_{idx}"
    else:
        log_text = case.get("text", "")
        if case.get("case_id"):
            case_id = case["case_id"]
        else:
            first_line = log_text.splitlines()[0].strip() if isinstance(log_text, str) else ""
            if first_line.startswith("Case-ID:"):
                case_id = first_line.replace("Case-ID:", "").strip()
            else:
                case_id = f"case_{idx}"

    prompt = build_prompt(log_text)

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content": "You extract structured JSON from maintenance logs."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            response_format={"type": "json_object"}
        )

        raw_output = response.choices[0].message.content

        try:
            parsed_output = json.loads(raw_output)
        except json.JSONDecodeError:
            parsed_output = None

        output_results.append({
            "case_id": case_id,
            "result": parsed_output
        })

    except Exception as e:
        output_results.append({
            "case_id": case_id,
            "result": None,
            "error": str(e)
        })

Processing case 1/20
Processing case 2/20
Processing case 3/20
Processing case 4/20
Processing case 5/20
Processing case 6/20
Processing case 7/20
Processing case 8/20
Processing case 9/20
Processing case 10/20
Processing case 11/20
Processing case 12/20
Processing case 13/20
Processing case 14/20
Processing case 15/20
Processing case 16/20
Processing case 17/20
Processing case 18/20
Processing case 19/20
Processing case 20/20


In [22]:
output_file = repo_root / "data" / "processed" / "prompt_only_extraction_gpt4o_20_cases_3.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(output_results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_file)

Saved to: c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\processed\prompt_only_extraction_gpt4o_20_cases_3.json


## Extraction with Qwen3 30B ONLY (via DEKA API)

In [42]:
import os

# Configure client for Qwen3 30B via Deka
deka_api_key = os.getenv("DEKA_API_KEY")
if deka_api_key is None:
    raise RuntimeError("DEKA_API_KEY not found. Please set it in your environment.")

client_qwen = OpenAI(
    base_url="https://dekallm.cloudeka.ai/v1",
    api_key=deka_api_key
)

print("Qwen3 30B client initialized with DEKA API")
print("API key loaded:", deka_api_key is not None)

Qwen3 30B client initialized with DEKA API
API key loaded: True


In [45]:
output_results_qwen = []

for idx, case in enumerate(cases, start=1):
    print(f"Processing case {idx}/{len(cases)}")

    if isinstance(case, str):
        log_text = case
        first_line = log_text.splitlines()[0].strip() if log_text else ""
        if first_line.startswith("Case-ID:"):
            case_id = first_line.replace("Case-ID:", "").strip()
        else:
            case_id = f"case_{idx}"
    else:
        log_text = case.get("text", "")
        if case.get("case_id"):
            case_id = case["case_id"]
        else:
            first_line = log_text.splitlines()[0].strip() if isinstance(log_text, str) else ""
            if first_line.startswith("Case-ID:"):
                case_id = first_line.replace("Case-ID:", "").strip()
            else:
                case_id = f"case_{idx}"

    prompt = build_prompt(log_text)

    try:
        response = client_qwen.chat.completions.create(
            model="qwen/qwen3-30b-a3b-instruct-2507",
            temperature=0,
            max_tokens=1000,
            messages=[
                {
                    "role": "system",
                    "content": "You extract structured JSON from maintenance logs."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        raw_output = response.choices[0].message.content

        try:
            parsed_output = json.loads(raw_output)
        except json.JSONDecodeError:
            parsed_output = None

        output_results_qwen.append({
            "case_id": case_id,
            "result": parsed_output
        })

    except Exception as e:
        output_results_qwen.append({
            "case_id": case_id,
            "result": None,
            "error": str(e)
        })

print(f"Completed {len(output_results_qwen)} cases")


Processing case 1/20
Processing case 2/20
Processing case 3/20
Processing case 4/20
Processing case 5/20
Processing case 6/20
Processing case 7/20
Processing case 8/20
Processing case 9/20
Processing case 10/20
Processing case 11/20
Processing case 12/20
Processing case 13/20
Processing case 14/20
Processing case 15/20
Processing case 16/20
Processing case 17/20
Processing case 18/20
Processing case 19/20
Processing case 20/20
Completed 20 cases


In [46]:
output_file_qwen = repo_root / "data" / "processed" / "prompt_only_extraction_qwen3_20_cases.json"

with open(output_file_qwen, "w", encoding="utf-8") as f:
    json.dump(output_results_qwen, f, indent=2, ensure_ascii=False)

print("Saved Qwen3 extraction to:", output_file_qwen)

Saved Qwen3 extraction to: c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\processed\prompt_only_extraction_qwen3_20_cases.json



Comparison : Prompt only + BAML + Qwen


In [48]:
def analyze_structure(result):
    """Validate result structure and count schema violations."""
    missing = extra = type_err = enum_err = 0
    if not isinstance(result, dict):
        return 5, 0, 5, 2

    # Top-level keys
    expected_top = {"fault_location", "fault_symptoms", "fault_reason", "fault_measures", "resolution_status"}
    for key in expected_top:
        if key not in result:
            missing += 1
    for key in result:
        if key not in expected_top:
            extra += 1

    # Validate fault_location
    loc = result.get("fault_location")
    if not isinstance(loc, dict):
        missing += 2
        type_err += 2
        enum_err += 1
    else:
        if "name" not in loc:
            missing += 1
        elif not isinstance(loc["name"], str):
            type_err += 1
        if "machine" not in loc:
            missing += 1
        else:
            m = loc["machine"]
            if m is not None and not isinstance(m, str):
                type_err += 1
            elif m not in {"IBM3", "IBM4"} and m is not None:
                enum_err += 1
        for key in loc:
            if key not in {"name", "machine"}:
                extra += 1

    # Validate fault_symptoms
    s = result.get("fault_symptoms")
    if s is None:
        missing += 1
    elif not isinstance(s, list):
        type_err += 1
    else:
        for item in s:
            if not isinstance(item, str):
                type_err += 1

    # Validate fault_reason
    r = result.get("fault_reason")
    if r is None:
        missing += 1
    elif not isinstance(r, list):
        type_err += 1
    else:
        for obj in r:
            if not isinstance(obj, dict) or "name" not in obj or not isinstance(obj.get("name"), str):
                type_err += 1
            if isinstance(obj, dict):
                for key in obj:
                    if key != "name":
                        extra += 1

    # Validate fault_measures
    m = result.get("fault_measures")
    if m is None:
        missing += 1
    elif not isinstance(m, list):
        type_err += 1
    else:
        for obj in m:
            if not isinstance(obj, dict) or "description" not in obj or not isinstance(obj.get("description"), str):
                type_err += 1
            if isinstance(obj, dict):
                for key in obj:
                    if key != "description":
                        extra += 1

    # Validate resolution_status
    rs = result.get("resolution_status")
    if rs is None:
        missing += 1
    elif not isinstance(rs, str):
        type_err += 1
    elif rs not in {"Resolved", "Unknown", "N/A"}:
        enum_err += 1

    return missing, extra, type_err, enum_err

def summarize_validation(data, label):
    """Validate and print stats for a dataset."""
    total_cases = len(data)
    total_missing = total_extra = total_type = total_enum = valid_cases = 0
    
    for item in data:
        missing, extra, type_err, enum_err = analyze_structure(item.get("result"))
        total_missing += missing
        total_extra += extra
        total_type += type_err
        total_enum += enum_err
        if missing == 0 and extra == 0 and type_err == 0 and enum_err == 0:
            valid_cases += 1
    
    print(f"\n{'='*60}")
    print(f"{label.upper()} - STRUCTURAL VALIDATION")
    print(f"{'='*60}")
    print(f"valid cases : {valid_cases} / {total_cases}")
    print(f"missing-field : {total_missing} / {total_cases * 5}")
    print(f"extra-field : {total_extra} / {total_cases * 5}")
    print(f"type error : {total_type} / {total_cases * 5}")
    print(f"enum violation : {total_enum} / {total_cases * 2}")
    
    return {
        "valid_cases": valid_cases,
        "total_cases": total_cases,
        "total_missing": total_missing,
        "total_extra": total_extra,
        "total_type": total_type,
        "total_enum": total_enum,
    }

# Load Qwen3 data
qwen_prompt_file = repo_root / "data" / "processed" / "prompt_only_extraction_qwen3_20_cases.json"
qwen_baml_file = repo_root / "data" / "processed" / "baml_extracted_20_cases_qwen3.json"

qwen_prompt_data = json.loads(qwen_prompt_file.read_text(encoding="utf-8"))
qwen_baml_data = json.loads(qwen_baml_file.read_text(encoding="utf-8"))
qwen_baml_by_id = {item["case_id"]: item["result"] for item in qwen_baml_data}

# Validate both approaches
qwen_prompt_stats = summarize_validation(qwen_prompt_data, "Qwen3 (prompt-only)")
qwen_baml_stats = summarize_validation(qwen_baml_data, "BAML+Qwen3")

# Compare outputs at case level
print(f"\n{'='*60}")
print("CASE-LEVEL DIFFERENCES (JSON equality)")
print(f"{'='*60}")

qwen_prompt_by_id = {item["case_id"]: item["result"] for item in qwen_prompt_data}

qwen_diff = sum(1 for cid in qwen_prompt_by_id if qwen_prompt_by_id[cid] != qwen_baml_by_id.get(cid))
print(f"Prompt-only vs BAML+Qwen3: {qwen_diff} / {len(qwen_prompt_by_id)} cases differ")

if qwen_diff > 0:
    for cid in qwen_prompt_by_id:
        if qwen_prompt_by_id[cid] != qwen_baml_by_id.get(cid):
            print(f"\nFirst diff sample - {cid}:")
            print(f"  Prompt-only: {qwen_prompt_by_id[cid]}")
            print(f"  BAML+Qwen3: {qwen_baml_by_id.get(cid)}")
            break

# Summary table
print(f"\n{'='*60}")
print("QWEN3 COMPARISON TABLE")
print(f"{'='*60}")
lines = []
lines.append('| Metric | Prompt-only | BAML+Qwen3 |')
lines.append('|---|---:|---:|')
lines.append(f"| valid cases | {qwen_prompt_stats['valid_cases']} / {qwen_prompt_stats['total_cases']} | {qwen_baml_stats['valid_cases']} / {qwen_baml_stats['total_cases']} |")
lines.append(f"| missing-field | {qwen_prompt_stats['total_missing']} / {qwen_prompt_stats['total_cases'] * 5} | {qwen_baml_stats['total_missing']} / {qwen_baml_stats['total_cases'] * 5} |")
lines.append(f"| extra-field | {qwen_prompt_stats['total_extra']} / {qwen_prompt_stats['total_cases'] * 5} | {qwen_baml_stats['total_extra']} / {qwen_baml_stats['total_cases'] * 5} |")
lines.append(f"| type error | {qwen_prompt_stats['total_type']} / {qwen_prompt_stats['total_cases'] * 5} | {qwen_baml_stats['total_type']} / {qwen_baml_stats['total_cases'] * 5} |")
lines.append(f"| enum violation | {qwen_prompt_stats['total_enum']} / {qwen_prompt_stats['total_cases'] * 2} | {qwen_baml_stats['total_enum']} / {qwen_baml_stats['total_cases'] * 2} |")
print('\n'.join(lines))


QWEN3 (PROMPT-ONLY) - STRUCTURAL VALIDATION
valid cases : 20 / 20
missing-field : 0 / 100
extra-field : 0 / 100
type error : 0 / 100
enum violation : 0 / 40

BAML+QWEN3 - STRUCTURAL VALIDATION
valid cases : 20 / 20
missing-field : 0 / 100
extra-field : 0 / 100
type error : 0 / 100
enum violation : 0 / 40

CASE-LEVEL DIFFERENCES (JSON equality)
Prompt-only vs BAML+Qwen3: 19 / 20 cases differ

First diff sample - IBM3_C15_23-Feb-15_23-Feb-15:
  Prompt-only: {'fault_location': {'name': 'cryocompressor', 'machine': 'IBM3'}, 'fault_symptoms': ['He-verlies/tekort in cryocompressor', 'Uit: 10,5bar / aan: 3,0 - 3,4bar (40-50 psi)', 'Displacer maakt een wat hoog piepend/schrapend geluid', 'Temp. Is 11K'], 'fault_reason': [], 'fault_measures': [{'description': "Absorber vervangen: 6-10-'06"}, {'description': 'CTI-Cryogenics'}, {'description': 'Helix technology Corp'}, {'description': '9600 compressor'}, {'description': 'P/N = 8135901G001'}, {'description': 'SN = 94L0013912'}], 'resolution_statu

## Comparison: Prompt-only vs BAML+GPT4

This cell prints a side-by-side conformance table and saves results to `data/processed/conformance_comparison.*`.

In [17]:
import json
from pathlib import Path

# Resolve notebook repo root if available; fall back to cwd
try:
    repo = repo_root
except NameError:
    repo = Path.cwd()

PROMPT_FILE = repo / "data" / "processed" / "prompt_only_extraction_gpt4o_20_cases_2.json"
BAML_FILE = repo / "data" / "processed" / "baml_extracted_20_cases.json"

print('Resolved prompt file:', PROMPT_FILE, 'exists=', PROMPT_FILE.exists())
print('Resolved baml file:', BAML_FILE, 'exists=', BAML_FILE.exists())

def ensure_and_load(path):
    if not path.exists():
        raise FileNotFoundError(f"Extraction file not found: {path}")
    return json.loads(path.read_text(encoding='utf-8'))

# quick existence check (will raise FileNotFoundError with helpful path if missing)
_ = ensure_and_load(PROMPT_FILE)
_ = ensure_and_load(BAML_FILE)

Resolved prompt file: c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\processed\prompt_only_extraction_gpt4o_20_cases_2.json exists= True
Resolved baml file: c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\processed\baml_extracted_20_cases.json exists= True


In [23]:
import json
from pathlib import Path

EXPECTED_FIELDS_PER_CASE = 6
ENUM_CHECKS_PER_CASE = 2

VALID_MACHINES = {"IBM3", "IBM4"}
VALID_RESOLUTION = {"Resolved", "Unknown", "N/A"}
EXPECTED_TOP_KEYS = {"fault_location", "fault_symptoms", "fault_reason", "fault_measures", "resolution_status"}
EXPECTED_LOCATION_KEYS = {"name", "machine"}

def analyze_result(result):
    missing = extra = type_err = enum_viol = 0
    if not isinstance(result, dict):
        return EXPECTED_FIELDS_PER_CASE, 0, EXPECTED_FIELDS_PER_CASE, 2
    for k in EXPECTED_TOP_KEYS:
        if k not in result:
            missing += 1
    for k in result:
        if k not in EXPECTED_TOP_KEYS:
            extra += 1
    loc = result.get("fault_location")
    if not isinstance(loc, dict):
        missing += 2; type_err += 2; enum_viol += 1
    else:
        if "name" not in loc: missing += 1
        elif not isinstance(loc["name"], str): type_err += 1
        if "machine" not in loc: missing += 1
        else:
            m = loc["machine"]
            if m is not None and not isinstance(m, str): type_err += 1
            elif m not in VALID_MACHINES and m is not None: enum_viol += 1
        for k in loc:
            if k not in EXPECTED_LOCATION_KEYS: extra += 1
    s = result.get("fault_symptoms")
    if s is None: missing += 1
    elif not isinstance(s, list): type_err += 1
    else:
        for item in s:
            if not isinstance(item, str): type_err += 1
    r = result.get("fault_reason")
    if r is None: missing += 1
    elif not isinstance(r, list): type_err += 1
    else:
        for obj in r:
            if not isinstance(obj, dict) or "name" not in obj or not isinstance(obj.get("name"), str): type_err += 1
            if isinstance(obj, dict):
                for k in obj:
                    if k != "name": extra += 1
    mlist = result.get("fault_measures")
    if mlist is None: missing += 1
    elif not isinstance(mlist, list): type_err += 1
    else:
        for obj in mlist:
            if not isinstance(obj, dict) or "description" not in obj or not isinstance(obj.get("description"), str): type_err += 1
            if isinstance(obj, dict):
                for k in obj:
                    if k != "description": extra += 1
    rs = result.get("resolution_status")
    if rs is None: missing += 1
    elif not isinstance(rs, str): type_err += 1
    elif rs not in VALID_RESOLUTION: enum_viol += 1
    return missing, extra, type_err, enum_viol

def summarize(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Extraction file not found: {p}")
    data = json.loads(p.read_text(encoding="utf-8"))
    n = len(data)
    total_expected = n * EXPECTED_FIELDS_PER_CASE
    total_enum_checks = n * ENUM_CHECKS_PER_CASE
    total_missing = total_extra = total_type_err = total_enum_viol = 0
    valid_cases = 0
    for item in data:
        result = item.get("result")
        missing, extra, type_err, enum_viol = analyze_result(result)
        total_missing += missing
        total_extra += extra
        total_type_err += type_err
        total_enum_viol += enum_viol
        if missing == 0 and extra == 0 and type_err == 0 and enum_viol == 0:
            valid_cases += 1
    print(f"valid cases : {valid_cases} / {n}")
    print(f"missing-field : {total_missing} / {total_expected}")
    print(f"extra-field : {total_extra} / {total_expected}")
    print(f"type error : {total_type_err} / {total_expected}")
    print(f"enum violation : {total_enum_viol} / {total_enum_checks}")
    return {'valid_cases': valid_cases, 'n': n, 'total_missing': total_missing, 'total_expected': total_expected, 'total_extra': total_extra, 'total_type_err': total_type_err, 'total_enum_viol': total_enum_viol, 'total_enum_checks': total_enum_checks}

# Direct notebook usage (no CLI argument parsing)
try:
    repo = repo_root
except NameError:
    repo = Path.cwd()

prompt_path = repo / "data" / "processed" / "prompt_only_extraction_gpt4o_20_cases_3.json"
baml_path = repo / "data" / "processed" / "baml_extracted_20_cases.json"

print("=" * 60)
print("PROMPT-ONLY EXTRACTION VALIDATION")
print("=" * 60)
prompt_stats = summarize(prompt_path)

print("\n" + "=" * 60)
print("BAML+GPT4 EXTRACTION VALIDATION")
print("=" * 60)
baml_stats = summarize(baml_path)

print("\n" + "=" * 60)
print("COMPARISON TABLE")
print("=" * 60)
lines = []
lines.append('| Metric | Prompt-only | BAML+GPT4 |')
lines.append('|---|---:|---:|')
lines.append(f"| valid cases | {prompt_stats['valid_cases']} / {prompt_stats['n']} | {baml_stats['valid_cases']} / {baml_stats['n']} |")
lines.append(f"| missing-field | {prompt_stats['total_missing']} / {prompt_stats['total_expected']} | {baml_stats['total_missing']} / {baml_stats['total_expected']} |")
lines.append(f"| extra-field | {prompt_stats['total_extra']} / {prompt_stats['total_expected']} | {baml_stats['total_extra']} / {baml_stats['total_expected']} |")
lines.append(f"| type error | {prompt_stats['total_type_err']} / {prompt_stats['total_expected']} | {baml_stats['total_type_err']} / {baml_stats['total_expected']} |")
lines.append(f"| enum violation | {prompt_stats['total_enum_viol']} / {prompt_stats['total_enum_checks']} | {baml_stats['total_enum_viol']} / {baml_stats['total_enum_checks']} |")
print('\n'.join(lines))

PROMPT-ONLY EXTRACTION VALIDATION
valid cases : 20 / 20
missing-field : 0 / 120
extra-field : 0 / 120
type error : 0 / 120
enum violation : 0 / 40

BAML+GPT4 EXTRACTION VALIDATION
valid cases : 20 / 20
missing-field : 0 / 120
extra-field : 0 / 120
type error : 0 / 120
enum violation : 0 / 40

COMPARISON TABLE
| Metric | Prompt-only | BAML+GPT4 |
|---|---:|---:|
| valid cases | 20 / 20 | 20 / 20 |
| missing-field | 0 / 120 | 0 / 120 |
| extra-field | 0 / 120 | 0 / 120 |
| type error | 0 / 120 | 0 / 120 |
| enum violation | 0 / 40 | 0 / 40 |
